In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

model_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning"

print(os.listdir(model_path))

['best_rnn_params.pkl', 'lstm_config.pkl', 'label_mapping.pkl', 'Data_cleaning.ipynb', 'requirements.txt', 'sent_valid.csv', 'streamlit.ipynb', 'sent_train.csv', 'cleaned_test.csv', 'best_gru_params.pkl', 'best_lstm_params.pkl', 'word_to_index.pkl', 'business_case_data.csv', 'models', 'cleaned.csv', 'lstm_model.pth', 'project_summary.docx', 'EDA.ipynb', 'rnn,lstm,gru.ipynb']


In [3]:
!pip install -q streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 110.2 MB/s eta 0:00:00


In [4]:
%%writefile app.py

# ============================================================
# FINANCIAL NEWS / TWEET SENTIMENT ANALYSIS
# LSTM + STREAMLIT BUSINESS APPLICATION
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import streamlit as st
import torch
import torch.nn as nn
import joblib
import os
import pandas as pd
import numpy as np


# ============================================================
# 2. PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Financial Sentiment Analytics",
    page_icon="📈",
    layout="wide"
)


# ============================================================
# 3. LSTM MODEL CLASS
# ============================================================

class LSTMModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        output_dim=3,
        dropout=0.3,
        bidirectional=False,
        padding_idx=0
    ):

        super().__init__()

        self.bidirectional = bidirectional

        # Word embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )

        # Dropout layer
        self.dropout = nn.Dropout(
            dropout
        )

        direction = (
            2 if bidirectional else 1
        )

        # Fully connected output layer
        self.fc = nn.Linear(
            hidden_dim * direction,
            output_dim
        )


    def forward(self, X):

        # Word embeddings
        X = self.embedding(X)

        # LSTM
        output, (hidden, cell) = (
            self.lstm(X)
        )

        # Handle bidirectional LSTM
        if self.bidirectional:

            hidden = torch.cat(
                (
                    hidden[-2],
                    hidden[-1]
                ),
                dim=1
            )

        else:

            hidden = hidden[-1]

        # Dropout
        hidden = self.dropout(
            hidden
        )

        # Output
        X = self.fc(hidden)

        return X


# ============================================================
# 4. MODEL LOCATION
# ============================================================

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "Financial_News_Sentiment_Prediction_using_DeepLearning"
)


# ============================================================
# 5. CHECK MODEL FILES
# ============================================================

required_files = [
    "word_to_index.pkl",
    "label_mapping.pkl",
    "lstm_config.pkl",
    "lstm_model.pth"
]


missing_files = []


for file_name in required_files:

    file_path = os.path.join(
        MODEL_PATH,
        file_name
    )

    if not os.path.exists(
        file_path
    ):

        missing_files.append(
            file_name
        )


if missing_files:

    st.error(
        "The following model files are missing:"
    )

    for file_name in missing_files:

        st.write(
            f"- {file_name}"
        )

    st.stop()


# ============================================================
# 6. LOAD SAVED FILES
# ============================================================

word_to_index = joblib.load(
    os.path.join(
        MODEL_PATH,
        "word_to_index.pkl"
    )
)


label_mapping = joblib.load(
    os.path.join(
        MODEL_PATH,
        "label_mapping.pkl"
    )
)


lstm_config = joblib.load(
    os.path.join(
        MODEL_PATH,
        "lstm_config.pkl"
    )
)


# ============================================================
# 7. CREATE LSTM MODEL
# ============================================================

lstm_model = LSTMModel(

    vocab_size=(
        lstm_config[
            "vocab_size"
        ]
    ),

    embedding_dim=(
        lstm_config[
            "embedding_dim"
        ]
    ),

    hidden_dim=(
        lstm_config[
            "hidden_dim"
        ]
    ),

    output_dim=(
        lstm_config[
            "output_dim"
        ]
    ),

    dropout=(
        lstm_config[
            "dropout"
        ]
    ),

    bidirectional=(
        lstm_config[
            "bidirectional"
        ]
    ),

    padding_idx=(
        lstm_config[
            "padding_idx"
        ]
    )
)


# ============================================================
# 8. LOAD TRAINED MODEL WEIGHTS
# ============================================================

lstm_model.load_state_dict(

    torch.load(

        os.path.join(
            MODEL_PATH,
            "lstm_model.pth"
        ),

        map_location="cpu"
    )
)


lstm_model.eval()


# ============================================================
# 9. TEXT TO SEQUENCE
# ============================================================

def text_to_sequence(text):

    return [

        word_to_index.get(
            word,
            word_to_index["<UNK>"]
        )

        for word in text.split()
    ]


# ============================================================
# 10. PREPARE TEXT
# ============================================================

def prepare_text(text):

    sequence = text_to_sequence(
        text
    )

    max_length = (
        lstm_config[
            "max_length"
        ]
    )

    # Truncate sequence
    sequence = (
        sequence[:max_length]
    )

    # Padding
    sequence = sequence + [

        word_to_index[
            "<PAD>"
        ]

    ] * (
        max_length
        -
        len(sequence)
    )

    return torch.tensor(
        [sequence],
        dtype=torch.long
    )


# ============================================================
# 11. PREDICT SENTIMENT
# ============================================================

def predict_sentiment(text):

    X = prepare_text(
        text
    )

    with torch.no_grad():

        output = lstm_model(
            X
        )

        probabilities = (
            torch.softmax(
                output,
                dim=1
            )
        )

        prediction = (
            torch.argmax(
                output,
                dim=1
            )
            .item()
        )

    sentiment = (
        label_mapping[
            prediction
        ]
    )

    confidence = (
        probabilities[
            0
        ][
            prediction
        ]
        .item()
    )

    probability_dict = {}


    for class_id, probability in enumerate(
        probabilities[0]
    ):

        class_name = (
            label_mapping[
                class_id
            ]
        )

        probability_dict[
            class_name
        ] = probability.item()


    return (
        sentiment,
        confidence,
        probability_dict
    )


# ============================================================
# 12. MAIN PAGE HEADER
# ============================================================

st.title(
    "📈 Financial News Sentiment Analytics"
)


st.write(
    "LSTM-based financial sentiment analysis "
    "and business intelligence application."
)


st.caption(
    "Sentiment Classes: "
    "Bullish | Neutral | Bearish"
)


# ============================================================
# 13. SINGLE TEXT SENTIMENT PREDICTION
# ============================================================

st.header(
    "🔍 Financial Text Sentiment Prediction"
)


text = st.text_area(

    "Enter financial news or tweet:",

    height=150,

    placeholder=(
        "Example: The company reported "
        "strong earnings and revenue growth."
    )
)


if st.button(
    "🔍 Predict Sentiment",
    type="primary"
):

    if text.strip() == "":

        st.warning(
            "Please enter some financial text."
        )

    else:

        (
            sentiment,
            confidence,
            probability_dict
        ) = predict_sentiment(
            text
        )


        st.subheader(
            "Prediction"
        )


        if sentiment == "Bullish":

            st.success(
                f"📈 {sentiment}"
            )

        elif sentiment == "Bearish":

            st.error(
                f"📉 {sentiment}"
            )

        else:

            st.info(
                f"➖ {sentiment}"
            )


        st.metric(
            "Prediction Confidence",
            f"{confidence:.2%}"
        )


        st.subheader(
            "Prediction Probabilities"
        )


        probability_df = pd.DataFrame(
            {
                "Sentiment":
                    list(
                        probability_dict.keys()
                    ),

                "Probability":
                    list(
                        probability_dict.values()
                    )
            }
        )


        probability_df[
            "Probability"
        ] = (

            probability_df[
                "Probability"
            ] * 100

        ).round(2)


        st.dataframe(
            probability_df,
            use_container_width=True,
            hide_index=True
        )


# ============================================================
# 14. BUSINESS USE CASE SECTION
# ============================================================

st.divider()


st.header(
    "💼 Financial Sentiment Business Applications"
)


st.write(
    "Upload financial tweets or news data "
    "to perform sentiment-based business analysis."
)


st.info(
    "Required column: text\n\n"
    "For all business features, the CSV should "
    "also contain date and ticker columns."
)


# ============================================================
# 15. CSV UPLOAD
# ============================================================

uploaded_file = st.file_uploader(

    "📂 Upload Financial Tweet / News CSV",

    type=[
        "csv"
    ]
)


if uploaded_file is not None:

    try:

        df = pd.read_csv(
            uploaded_file
        )

    except Exception as e:

        st.error(
            f"Unable to read CSV: {e}"
        )

        st.stop()


    # ========================================================
    # 16. VALIDATE TEXT COLUMN
    # ========================================================

    if "text" not in df.columns:

        st.error(
            "CSV must contain a column "
            "named 'text'."
        )

        st.write(
            "Available columns:"
        )

        st.write(
            list(df.columns)
        )

        st.stop()


    st.success(
        f"Successfully loaded "
        f"{len(df):,} records."
    )


    # ========================================================
    # 17. BATCH SENTIMENT PREDICTION
    # ========================================================

    sentiments = []

    confidences = []

    bullish_probabilities = []

    bearish_probabilities = []

    neutral_probabilities = []


    progress_bar = st.progress(
        0
    )


    total_rows = len(df)


    for i, text_value in enumerate(
        df["text"].fillna("")
    ):

        text_value = str(
            text_value
        ).strip()


        if text_value == "":

            sentiments.append(
                "Neutral"
            )

            confidences.append(
                0.0
            )

            bullish_probabilities.append(
                0.0
            )

            bearish_probabilities.append(
                0.0
            )

            neutral_probabilities.append(
                0.0
            )


        else:

            (
                sentiment,
                confidence,
                probability_dict
            ) = predict_sentiment(
                text_value
            )


            sentiments.append(
                sentiment
            )


            confidences.append(
                confidence
            )


            bullish_probabilities.append(

                probability_dict.get(
                    "Bullish",
                    0.0
                )
            )


            bearish_probabilities.append(

                probability_dict.get(
                    "Bearish",
                    0.0
                )
            )


            neutral_probabilities.append(

                probability_dict.get(
                    "Neutral",
                    0.0
                )
            )


        progress_bar.progress(
            (i + 1)
            /
            total_rows
        )


    progress_bar.empty()


    # ========================================================
    # 18. ADD MODEL RESULTS TO DATAFRAME
    # ========================================================

    df[
        "sentiment"
    ] = sentiments


    df[
        "confidence"
    ] = confidences


    df[
        "bullish_probability"
    ] = bullish_probabilities


    df[
        "bearish_probability"
    ] = bearish_probabilities


    df[
        "neutral_probability"
    ] = neutral_probabilities


    # ========================================================
    # 19. PREPARE DATE COLUMN
    # ========================================================

    has_date = (
        "date" in df.columns
    )


    has_ticker = (
        "ticker" in df.columns
    )


    if has_date:

        df[
            "date"
        ] = pd.to_datetime(
            df["date"],
            errors="coerce"
        )

        df[
            "day"
        ] = (
            df["date"]
            .dt.date
        )


    # ========================================================
    # 20. BUSINESS CASE NAVIGATION
    # ========================================================

    st.divider()


    business_case = st.radio(

        "Select Business Application",

        [
            "1. Retail Investor Sentiment",
            "2. Portfolio Risk Monitoring",
            "3. Trading Signal Prototype",
            "4. Content Triage"
        ],

        horizontal=True
    )


    # ========================================================
    # BUSINESS CASE 1
    # RETAIL INVESTOR SENTIMENT DASHBOARD
    # ========================================================

    if business_case == (
        "1. Retail Investor Sentiment"
    ):

        st.subheader(
            "📊 Retail Investor Sentiment Dashboard"
        )


        st.write(
            "Track Bullish, Bearish, and Neutral "
            "financial sentiment to understand "
            "overall market mood."
        )


        dashboard_df = (
            df.copy()
        )


        # ----------------------------------------------------
        # Ticker filter
        # ----------------------------------------------------

        if has_ticker:

            tickers = sorted(

                dashboard_df[
                    "ticker"
                ]
                .dropna()
                .astype(str)
                .unique()
            )


            selected_tickers = (
                st.multiselect(

                    "Select Ticker(s)",

                    tickers,

                    default=(
                        tickers[:5]
                    )
                )
            )


            if selected_tickers:

                dashboard_df = (
                    dashboard_df[
                        dashboard_df[
                            "ticker"
                        ]
                        .astype(str)
                        .isin(
                            selected_tickers
                        )
                    ]
                )


        # ----------------------------------------------------
        # Sentiment counts
        # ----------------------------------------------------

        sentiment_counts = (

            dashboard_df[
                "sentiment"
            ]
            .value_counts()
            .reindex(
                [
                    "Bullish",
                    "Neutral",
                    "Bearish"
                ],
                fill_value=0
            )
        )


        total_tweets = (
            len(
                dashboard_df
            )
        )


        bullish_count = int(
            sentiment_counts[
                "Bullish"
            ]
        )


        bearish_count = int(
            sentiment_counts[
                "Bearish"
            ]
        )


        neutral_count = int(
            sentiment_counts[
                "Neutral"
            ]
        )


        # ----------------------------------------------------
        # KPI metrics
        # ----------------------------------------------------

        col1, col2, col3, col4 = (
            st.columns(4)
        )


        with col1:

            st.metric(
                "Total Tweets",
                f"{total_tweets:,}"
            )


        with col2:

            st.metric(
                "📈 Bullish",
                f"{bullish_count:,}"
            )


        with col3:

            st.metric(
                "📉 Bearish",
                f"{bearish_count:,}"
            )


        with col4:

            st.metric(
                "➖ Neutral",
                f"{neutral_count:,}"
            )


        # ----------------------------------------------------
        # Overall sentiment chart
        # ----------------------------------------------------

        st.write(
            "### Overall Sentiment Distribution"
        )


        st.bar_chart(
            sentiment_counts
        )


        # ----------------------------------------------------
        # Daily sentiment
        # ----------------------------------------------------

        if has_date:

            daily_sentiment = (

                dashboard_df
                .dropna(
                    subset=[
                        "day"
                    ]
                )
                .groupby(
                    [
                        "day",
                        "sentiment"
                    ]
                )
                .size()
                .unstack(
                    fill_value=0
                )
                .reindex(
                    columns=[
                        "Bullish",
                        "Neutral",
                        "Bearish"
                    ],
                    fill_value=0
                )
            )


            st.write(
                "### 📅 Daily Sentiment Volume"
            )


            if not daily_sentiment.empty:

                st.line_chart(
                    daily_sentiment
                )

            else:

                st.info(
                    "No valid dates are available."
                )

        else:

            st.info(
                "Add a 'date' column to display "
                "daily sentiment trends."
            )


        # ----------------------------------------------------
        # Sentiment by ticker
        # ----------------------------------------------------

        if has_ticker:

            st.write(
                "### Sentiment by Ticker"
            )


            ticker_sentiment = (

                df.groupby(
                    [
                        "ticker",
                        "sentiment"
                    ]
                )
                .size()
                .unstack(
                    fill_value=0
                )
                .reindex(
                    columns=[
                        "Bullish",
                        "Neutral",
                        "Bearish"
                    ],
                    fill_value=0
                )
            )


            st.dataframe(
                ticker_sentiment,
                use_container_width=True
            )


    # ========================================================
    # BUSINESS CASE 2
    # PORTFOLIO RISK MONITORING
    # ========================================================

    elif business_case == (
        "2. Portfolio Risk Monitoring"
    ):

        st.subheader(
            "⚠️ Portfolio Risk Monitoring"
        )


        st.write(
            "Monitor sudden increases in Bearish "
            "sentiment around companies or tickers "
            "to support risk-review discussions."
        )


        if not has_ticker:

            st.warning(
                "A 'ticker' column is required "
                "for portfolio risk monitoring."
            )

        else:

            risk_df = (
                df.copy()
            )


            risk_df[
                "is_bearish"
            ] = (

                risk_df[
                    "sentiment"
                ]
                == "Bearish"

            ).astype(int)


            # ------------------------------------------------
            # Alert threshold
            # ------------------------------------------------

            risk_threshold = (
                st.slider(

                    "Bearish Sentiment Alert Threshold (%)",

                    min_value=10,

                    max_value=100,

                    value=60,

                    step=5
                )
            )


            # ------------------------------------------------
            # Daily risk monitoring
            # ------------------------------------------------

            if has_date:

                daily_risk = (

                    risk_df
                    .dropna(
                        subset=[
                            "day"
                        ]
                    )
                    .groupby(
                        [
                            "day",
                            "ticker"
                        ]
                    )
                    .agg(

                        total_tweets=(
                            "sentiment",
                            "count"
                        ),

                        bearish_tweets=(
                            "is_bearish",
                            "sum"
                        ),

                        average_bearish_probability=(
                            "bearish_probability",
                            "mean"
                        )
                    )
                    .reset_index()
                )


                daily_risk[
                    "bearish_percentage"
                ] = (

                    daily_risk[
                        "bearish_tweets"
                    ]

                    /

                    daily_risk[
                        "total_tweets"
                    ]

                    * 100
                )


                daily_risk[
                    "risk_status"
                ] = np.where(

                    daily_risk[
                        "bearish_percentage"
                    ]
                    >=
                    risk_threshold,

                    "Review Required",

                    "Normal"
                )


                alerts = (

                    daily_risk[
                        daily_risk[
                            "risk_status"
                        ]
                        ==
                        "Review Required"
                    ]
                    .sort_values(
                        "bearish_percentage",
                        ascending=False
                    )
                )


                # --------------------------------------------
                # Alert KPIs
                # --------------------------------------------

                col1, col2 = (
                    st.columns(2)
                )


                with col1:

                    st.metric(
                        "Risk Alerts",
                        f"{len(alerts):,}"
                    )


                with col2:

                    st.metric(
                        "Threshold",
                        f"{risk_threshold}%"
                    )


                st.write(
                    "### Risk Review Alerts"
                )


                if alerts.empty:

                    st.success(
                        "No ticker exceeds the selected "
                        "bearish sentiment threshold."
                    )

                else:

                    st.warning(
                        f"{len(alerts):,} risk-review "
                        "alert(s) detected."
                    )


                    st.dataframe(
                        alerts,
                        use_container_width=True,
                        hide_index=True
                    )


                # --------------------------------------------
                # Ticker risk trend
                # --------------------------------------------

                risk_tickers = sorted(

                    daily_risk[
                        "ticker"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )


                if risk_tickers:

                    selected_risk_ticker = (
                        st.selectbox(

                            "Select Ticker for Risk Trend",

                            risk_tickers
                        )
                    )


                    ticker_risk = (

                        daily_risk[
                            daily_risk[
                                "ticker"
                            ]
                            .astype(str)
                            ==
                            selected_risk_ticker
                        ]
                        .sort_values(
                            "day"
                        )
                    )


                    st.write(
                        "### Daily Bearish Sentiment"
                    )


                    risk_chart = (

                        ticker_risk[
                            [
                                "day",
                                "bearish_percentage"
                            ]
                        ]
                        .set_index(
                            "day"
                        )
                    )


                    st.line_chart(
                        risk_chart
                    )


            # ------------------------------------------------
            # Overall ticker risk if no date
            # ------------------------------------------------

            else:

                risk_table = (

                    risk_df
                    .groupby(
                        "ticker"
                    )
                    .agg(

                        total_mentions=(
                            "sentiment",
                            "count"
                        ),

                        bearish_mentions=(
                            "is_bearish",
                            "sum"
                        )
                    )
                    .reset_index()
                )


                risk_table[
                    "bearish_percentage"
                ] = (

                    risk_table[
                        "bearish_mentions"
                    ]

                    /

                    risk_table[
                        "total_mentions"
                    ]

                    * 100
                )


                risk_table[
                    "risk_status"
                ] = np.where(

                    risk_table[
                        "bearish_percentage"
                    ]
                    >=
                    risk_threshold,

                    "Review Required",

                    "Normal"
                )


                st.dataframe(
                    risk_table.sort_values(
                        "bearish_percentage",
                        ascending=False
                    ),
                    use_container_width=True,
                    hide_index=True
                )


                st.info(
                    "Add a 'date' column to monitor "
                    "daily bearish sentiment changes."
                )


    # ========================================================
    # BUSINESS CASE 3
    # NEWS-DRIVEN TRADING SIGNAL PROTOTYPE
    # ========================================================

    elif business_case == (
        "3. Trading Signal Prototype"
    ):

        st.subheader(
            "🔄 News-Driven Trading Signal Prototype"
        )


        st.warning(
            "Prototype for research and backtesting only. "
            "Sentiment signals are not trading "
            "recommendations."
        )


        if not has_ticker or not has_date:

            st.warning(
                "Both 'ticker' and 'date' columns "
                "are required for time-based "
                "sentiment signals."
            )

        else:

            signal_df = (
                df.copy()
            )


            signal_df = (
                signal_df.dropna(
                    subset=[
                        "date",
                        "ticker"
                    ]
                )
            )


            # ------------------------------------------------
            # Sentiment score
            # ------------------------------------------------

            sentiment_score_mapping = {

                "Bearish": -1,

                "Neutral": 0,

                "Bullish": 1
            }


            signal_df[
                "sentiment_score"
            ] = (

                signal_df[
                    "sentiment"
                ]
                .map(
                    sentiment_score_mapping
                )
            )


            # ------------------------------------------------
            # Ticker selector
            # ------------------------------------------------

            signal_tickers = sorted(

                signal_df[
                    "ticker"
                ]
                .astype(str)
                .unique()
            )


            selected_signal_ticker = (
                st.selectbox(

                    "Select Ticker",

                    signal_tickers,

                    key="signal_ticker"
                )
            )


            ticker_signal_df = (

                signal_df[
                    signal_df[
                        "ticker"
                    ]
                    .astype(str)
                    ==
                    selected_signal_ticker
                ]
                .copy()
            )


            # ------------------------------------------------
            # Daily aggregate sentiment
            # ------------------------------------------------

            daily_score = (

                ticker_signal_df
                .groupby(
                    "day"
                )
                .agg(

                    sentiment_score=(
                        "sentiment_score",
                        "mean"
                    ),

                    tweet_count=(
                        "sentiment",
                        "count"
                    ),

                    bullish_probability=(
                        "bullish_probability",
                        "mean"
                    ),

                    bearish_probability=(
                        "bearish_probability",
                        "mean"
                    )
                )
                .reset_index()
                .sort_values(
                    "day"
                )
            )


            # ------------------------------------------------
            # Convert score to daily sentiment
            # ------------------------------------------------

            daily_score[
                "daily_sentiment"
            ] = np.select(

                [
                    daily_score[
                        "sentiment_score"
                    ] > 0,

                    daily_score[
                        "sentiment_score"
                    ] < 0
                ],

                [
                    "Bullish",
                    "Bearish"
                ],

                default="Neutral"
            )


            # ------------------------------------------------
            # Previous sentiment
            # ------------------------------------------------

            daily_score[
                "previous_sentiment"
            ] = (

                daily_score[
                    "daily_sentiment"
                ]
                .shift(1)
            )


            # ------------------------------------------------
            # Default signal
            # ------------------------------------------------

            daily_score[
                "signal"
            ] = "No Signal"


            # ------------------------------------------------
            # Bearish -> Bullish
            # ------------------------------------------------

            bullish_flip = (

                (
                    daily_score[
                        "previous_sentiment"
                    ]
                    == "Bearish"
                )

                &

                (
                    daily_score[
                        "daily_sentiment"
                    ]
                    == "Bullish"
                )
            )


            # ------------------------------------------------
            # Bullish -> Bearish
            # ------------------------------------------------

            bearish_flip = (

                (
                    daily_score[
                        "previous_sentiment"
                    ]
                    == "Bullish"
                )

                &

                (
                    daily_score[
                        "daily_sentiment"
                    ]
                    == "Bearish"
                )
            )


            daily_score.loc[
                bullish_flip,
                "signal"
            ] = (
                "Bearish → Bullish"
            )


            daily_score.loc[
                bearish_flip,
                "signal"
            ] = (
                "Bullish → Bearish"
            )


            # ------------------------------------------------
            # Current sentiment
            # ------------------------------------------------

            if not daily_score.empty:

                latest = (
                    daily_score.iloc[-1]
                )


                col1, col2, col3 = (
                    st.columns(3)
                )


                with col1:

                    st.metric(
                        "Latest Sentiment",
                        latest[
                            "daily_sentiment"
                        ]
                    )


                with col2:

                    st.metric(
                        "Sentiment Score",
                        f"{latest['sentiment_score']:.3f}"
                    )


                with col3:

                    st.metric(
                        "Tweets Analysed",
                        int(
                            latest[
                                "tweet_count"
                            ]
                        )
                    )


            # ------------------------------------------------
            # Daily sentiment score chart
            # ------------------------------------------------

            st.write(
                f"### Daily Sentiment Score - "
                f"{selected_signal_ticker}"
            )


            sentiment_chart = (

                daily_score[
                    [
                        "day",
                        "sentiment_score"
                    ]
                ]
                .set_index(
                    "day"
                )
            )


            st.line_chart(
                sentiment_chart
            )


            # ------------------------------------------------
            # Sentiment flip signals
            # ------------------------------------------------

            detected_signals = (

                daily_score[
                    daily_score[
                        "signal"
                    ]
                    !=
                    "No Signal"
                ]
            )


            st.write(
                "### Detected Sentiment Flips"
            )


            if detected_signals.empty:

                st.info(
                    "No Bullish/Bearish sentiment "
                    "flips were detected."
                )

            else:

                st.dataframe(
                    detected_signals,
                    use_container_width=True,
                    hide_index=True
                )


            # ------------------------------------------------
            # Daily data
            # ------------------------------------------------

            with st.expander(
                "View Daily Signal Data"
            ):

                st.dataframe(
                    daily_score,
                    use_container_width=True,
                    hide_index=True
                )


    # ========================================================
    # BUSINESS CASE 4
    # CONTENT TRIAGE
    # ========================================================

    elif business_case == (
        "4. Content Triage"
    ):

        st.subheader(
            "📑 Content Triage for Research Teams"
        )


        st.write(
            "Automatically prioritize highly Bullish "
            "or Bearish financial content so analysts "
            "can review the most important items first."
        )


        # ----------------------------------------------------
        # Sentiment selection
        # ----------------------------------------------------

        selected_sentiments = (
            st.multiselect(

                "Select Sentiment to Review",

                [
                    "Bullish",
                    "Bearish"
                ],

                default=[
                    "Bullish",
                    "Bearish"
                ]
            )
        )


        # ----------------------------------------------------
        # Confidence threshold
        # ----------------------------------------------------

        confidence_threshold = (
            st.slider(

                "Minimum Prediction Confidence",

                min_value=0.00,

                max_value=1.00,

                value=0.70,

                step=0.05
            )
        )


        # ----------------------------------------------------
        # Filter records
        # ----------------------------------------------------

        triage_df = (

            df[
                (
                    df[
                        "sentiment"
                    ]
                    .isin(
                        selected_sentiments
                    )
                )

                &

                (
                    df[
                        "confidence"
                    ]
                    >=
                    confidence_threshold
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Assign priority
        # ----------------------------------------------------

        triage_df[
            "priority"
        ] = np.select(

            [
                triage_df[
                    "confidence"
                ] >= 0.90,

                triage_df[
                    "confidence"
                ] >= 0.80
            ],

            [
                "High",
                "Medium"
            ],

            default="Review"
        )


        # ----------------------------------------------------
        # Sort by confidence
        # ----------------------------------------------------

        triage_df = (

            triage_df
            .sort_values(
                "confidence",
                ascending=False
            )
        )


        # ----------------------------------------------------
        # KPI metrics
        # ----------------------------------------------------

        high_priority = (

            triage_df[
                "priority"
            ]
            == "High"

        ).sum()


        bullish_priority = (

            triage_df[
                "sentiment"
            ]
            == "Bullish"

        ).sum()


        bearish_priority = (

            triage_df[
                "sentiment"
            ]
            == "Bearish"

        ).sum()


        col1, col2, col3, col4 = (
            st.columns(4)
        )


        with col1:

            st.metric(
                "Tweets for Review",
                f"{len(triage_df):,}"
            )


        with col2:

            st.metric(
                "High Priority",
                f"{high_priority:,}"
            )


        with col3:

            st.metric(
                "Bullish",
                f"{bullish_priority:,}"
            )


        with col4:

            st.metric(
                "Bearish",
                f"{bearish_priority:,}"
            )


        # ----------------------------------------------------
        # Display columns
        # ----------------------------------------------------

        display_columns = [

            column

            for column in [

                "date",
                "ticker",
                "text",
                "sentiment",
                "confidence",
                "priority"

            ]

            if column
            in triage_df.columns
        ]


        st.write(
            "### Priority Content for Analyst Review"
        )


        if triage_df.empty:

            st.info(
                "No records match the selected "
                "sentiment and confidence threshold."
            )

        else:

            display_df = (

                triage_df[
                    display_columns
                ]
                .copy()
            )


            display_df[
                "confidence"
            ] = (

                display_df[
                    "confidence"
                ]

                * 100

            ).round(2)


            st.dataframe(
                display_df,
                use_container_width=True,
                hide_index=True
            )


    # ========================================================
    # 21. BUSINESS APPLICATION SUMMARY
    # ========================================================

    st.divider()


    st.subheader(
        "💼 Business Application Summary"
    )


    summary_df = pd.DataFrame(
        {
            "Business Case": [

                "Retail Investor Sentiment",

                "Portfolio Risk Monitoring",

                "Trading Signal Prototype",

                "Content Triage"
            ],

            "Purpose": [

                (
                    "Track daily Bullish, Bearish "
                    "and Neutral market sentiment."
                ),

                (
                    "Identify high Bearish sentiment "
                    "around individual companies."
                ),

                (
                    "Detect Bearish-to-Bullish and "
                    "Bullish-to-Bearish sentiment changes."
                ),

                (
                    "Prioritize strongly Bullish or "
                    "Bearish content for analyst review."
                )
            ]
        }
    )


    st.dataframe(
        summary_df,
        use_container_width=True,
        hide_index=True
    )


    # ========================================================
    # 22. DOWNLOAD PREDICTIONS
    # ========================================================

    st.divider()


    st.subheader(
        "📥 Download Sentiment Results"
    )


    download_df = (
        df.copy()
    )


    # Remove helper day column
    if "day" in download_df.columns:

        download_df = (
            download_df.drop(
                columns=[
                    "day"
                ]
            )
        )


    csv_data = (

        download_df
        .to_csv(
            index=False
        )
        .encode(
            "utf-8"
        )
    )


    st.download_button(

        label=(
            "Download Predictions CSV"
        ),

        data=csv_data,

        file_name=(
            "financial_sentiment_predictions.csv"
        ),

        mime="text/csv"
    )

Writing app.py


In [5]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-10-06 05:09:58--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64 [following]
--2026-10-06 05:09:59--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/96692278-72f4-40bc-b493-dbd69628f37a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-06T06%3A03%3A03Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-06

In [6]:
!streamlit run /content/app.py &>/content/logs.txt &

In [7]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://spirituality-nose-treatment-dust.trycloudflare.com


The company reported record profits and strong revenue growth this quarter.
The company announced a significant decline in revenue and profits.
The company reported quarterly earnings in line with market expectations.

In [ ]:
%%writefile /content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/requirements.txt

streamlit
torch
joblib
nltk
scikit-learn

Overwriting /content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/requirements.txt
